In [10]:
import csv
import pandas as pd

clean_rows = []

with open("crpc_pure_raw.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    # Read header (PART, ARTICLE, CONTENT or similar 3-column setup)
    header = next(reader)

    for line in reader:
        # Strip empty entries or trailing spaces from fields
        line = [field.strip() for field in line if field.strip() != ""]

        if not line:
            continue

        # Ensure we have at least PART and ARTICLE
        if len(line) >= 3:
            part = line[0]
            # Handle cases where PART name might have split into 2 parts
            # e.g., ['CHAPTER I', 'INTRODUCTION', '3', 'Punishment...']
            if not line[2].isdigit() and len(line) > 3:
                # Merge CHAPTER + CHAPTER_NAME if split across indices 0 and 1
                part = f"{line[0]}, {line[1]}"
                article = line[2]
                content = ", ".join(line[3:])
            else:
                article = line[1]
                content = ", ".join(line[2:])

            clean_rows.append(
                {"PART": part, "ARTICLE": article, "CONTENT": content}
            )

# Convert cleaned records to DataFrame
df_raw = pd.DataFrame(clean_rows)

In [11]:
import re


def parse_part(part_str):
    if not isinstance(part_str, str):
        return "", ""

    # Match "CHAPTER <Num>" followed by optional separator (comma, dash, space)
    match = re.match(r"^(CHAPTER\s+[IVXLCDM0-9]+)[\s,–\-]+(.*)$", part_str, re.IGNORECASE)

    if match:
        chapter_num = match.group(1).strip()
        chapter_name = match.group(2).strip()
        return chapter_num, chapter_name
    else:
        # Fallback if pattern doesn't match standard CHAPTER prefix
        return part_str.strip(), ""


# Apply extraction
parsed_parts = df_raw["PART"].apply(parse_part)
df_raw["Chapter"] = [p[0] for p in parsed_parts]
df_raw["Chapter_name"] = [p[1] for p in parsed_parts]

In [ ]:
def parse_article(article_str):
    if not isinstance(article_str, str):
        return "", ""

    # Case A: Article starts with number/code followed by separator (. or - or spaces) and title
    # e.g., "1 – Short title, extent", "2. Definitions."
    match = re.match(
        r"^([0-9]+[A-Z]?)\s*[\.\–\-]\s*(.*)$", article_str.strip()
    )

    if match:
        section_num = match.group(1).strip()
        section_name = match.group(2).strip()
        return section_num, section_name

    # Case B: Article is just the Section code (e.g., "138A", "295A")
    # In your sample data, section title is sometimes located in CONTENT
    return article_str.strip(), ""


parsed_articles = df_raw["ARTICLE"].apply(parse_article)
df_raw["Section"] = [a[0] for a in parsed_articles]
df_raw["Section_name"] = [a[1] for a in parsed_articles]

# Rename CONTENT to Description
df_raw["Description"] = df_raw["CONTENT"]

In [13]:
# Reorder attributes into final structure
output_cols = [
    "Chapter",
    "Chapter_name",
    "Section",
    "Section_name",
    "Description",
]
df_final = df_raw[output_cols]

# Save output CSV
df_final.to_csv("crpc_raw.csv", index=False, encoding="utf-8")

print("Conversion complete! First 5 rows:")
print(df_final.head())

KeyError: "['Section', 'Section_name', 'Description'] not in index"

Saved successfully! Current columns: ['Chapter', 'Chapter_name', 'Section', 'Section_name', 'Description']


In [1]:
import pandas as pd

df = pd.read_csv("crpc_raw.csv")

# Save the description as a single CSV field
df.to_csv("crpc_raw.csv", index=False, encoding="utf-8")

ParserError: Error tokenizing data. C error: Expected 5 fields in line 222, saw 40
